# Comparação — BRICS AI Governance Declaration × OECD Recommendation on AI

Este notebook compara os dois documentos já extraídos (`BRICS.json` e
`OCDE.json`) em dois eixos:

1. **Similaridade textual** — o quanto o vocabulário e a ênfase temática dos
   dois documentos se sobrepõem, usando o índice de Jaccard (sobreposição de
   vocabulário) e a similaridade de cosseno (sobreposição ponderada por
   frequência dos termos);
2. **Divergência/convergência por termo** — quais termos cada documento
   prioriza mais (gráfico *tornado*) e como os termos compartilhados se
   distribuem entre os dois documentos (dispersão BRICS × OCDE).

## Importação, carregamento e tokenização

In [ ]:
import json
import math
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

STOPWORDS = {
    "the", "a", "an", "and", "of", "to", "in", "that", "for", "as", "on",
    "with", "by", "or", "be", "is", "are", "we", "our", "their", "all",
    "at", "from", "this", "these", "those", "such", "which", "it", "its",
    "within", "while", "also", "both", "other", "than", "then", "so",
    "but", "not", "no", "do", "does", "did", "has", "have", "had", "was",
    "were", "been", "being", "will", "would", "should", "could", "can",
    "may", "might", "must", "shall", "us", "any", "into", "across", "over",
    "under", "about", "between", "including", "particularly", "especially",
    "towards", "toward", "through", "among", "who", "what", "how", "if",
    "each", "more", "most", "some", "own", "they", "them", "there",
}

def load_tokens(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    text = " ".join(section["text"] for section in data["sections"])
    tokens = re.findall(r"[a-zA-Z]+(?:-[a-zA-Z]+)*", text.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2], data

BRICS_LABEL = "BRICS Declaration"
OECD_LABEL = "OECD Recommendation"

brics_tokens, brics_data = load_tokens("BRICS.json")
ocde_tokens, ocde_data = load_tokens("OCDE.json")

brics_counts = Counter(brics_tokens)
ocde_counts = Counter(ocde_tokens)

print(f"{BRICS_LABEL}: {len(brics_tokens)} tokens, {len(brics_counts)} termos únicos")
print(f"{OECD_LABEL}: {len(ocde_tokens)} tokens, {len(ocde_counts)} termos únicos")


## 1. Similaridade entre os textos

Duas métricas complementares:

- **Índice de Jaccard** — proporção de termos que aparecem nos *dois*
  documentos em relação ao total de termos distintos usados por qualquer um
  deles. Mede sobreposição pura de vocabulário, sem considerar frequência.
- **Similaridade de cosseno** — trata cada documento como um vetor de
  frequências relativas (por 1.000 tokens) sobre o vocabulário combinado.
  Mede o quanto os dois documentos "apontam na mesma direção" temática,
  dando mais peso a termos usados com frequência semelhante nos dois.

In [ ]:
def relative_freq(counter, total_tokens, term):
    return counter.get(term, 0) / total_tokens * 1000

vocab_brics = set(brics_counts)
vocab_ocde = set(ocde_counts)
vocab_union = vocab_brics | vocab_ocde
vocab_intersection = vocab_brics & vocab_ocde

jaccard = len(vocab_intersection) / len(vocab_union)

terms = sorted(vocab_union)
vec_brics = [relative_freq(brics_counts, len(brics_tokens), t) for t in terms]
vec_ocde = [relative_freq(ocde_counts, len(ocde_tokens), t) for t in terms]

dot = sum(a * b for a, b in zip(vec_brics, vec_ocde))
norm_brics = math.sqrt(sum(a * a for a in vec_brics))
norm_ocde = math.sqrt(sum(b * b for b in vec_ocde))
cosine_similarity = dot / (norm_brics * norm_ocde)

print(f"Vocabulário BRICS:        {len(vocab_brics)} termos")
print(f"Vocabulário OCDE:         {len(vocab_ocde)} termos")
print(f"Termos em comum:         {len(vocab_intersection)} termos")
print(f"Índice de Jaccard:       {jaccard:.3f}  ({jaccard*100:.1f}% do vocabulário combinado é compartilhado)")
print(f"Similaridade de cosseno: {cosine_similarity:.3f}  (1 = idêntico, 0 = nenhuma sobreposição de ênfase)")


### Gráfico — as duas métricas de similaridade lado a lado

Ambas as métricas ficam entre 0 e 1, o que permite compará-las diretamente
na mesma escala.

In [ ]:
SEQUENTIAL_BLUE = "#2a78d6"
INK_PRIMARY = "#0b0b0b"
INK_MUTED = "#898781"
GRIDLINE = "#e1e0d9"
SURFACE = "#fcfcfb"

def plot_ranked_bar(pairs, title, xlabel="Frequência (nº de ocorrências)", xlim_max=None, value_fmt="{:.0f}", filename=None):
    labels = [p[0] for p in pairs][::-1]
    values = [p[1] for p in pairs][::-1]

    fig, ax = plt.subplots(figsize=(8, max(2.2, 0.45 * len(pairs))), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)

    bars = ax.barh(labels, values, color=SEQUENTIAL_BLUE, height=0.55, zorder=3)

    ax.set_xlabel(xlabel, color=INK_MUTED, fontsize=10)
    ax.set_title(title, color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

    ax.tick_params(axis="y", colors=INK_PRIMARY, labelsize=10)
    ax.tick_params(axis="x", colors=INK_MUTED, labelsize=9)

    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(GRIDLINE)

    ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

    max_val = xlim_max if xlim_max is not None else max(values)
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_width() + max_val * 0.015,
            bar.get_y() + bar.get_height() / 2,
            value_fmt.format(value),
            va="center", ha="left",
            color=INK_PRIMARY, fontsize=9,
        )
    ax.set_xlim(0, max_val * 1.15)

    fig.tight_layout()
    if filename:
        fig.savefig(filename, dpi=150, facecolor=SURFACE, bbox_inches="tight")
    plt.show()

plot_ranked_bar(
    [("Índice de Jaccard\n(sobreposição de vocabulário)", jaccard),
     ("Similaridade de cosseno\n(sobreposição de ênfase)", cosine_similarity)],
    "Similaridade textual — BRICS Declaration × OECD Recommendation",
    xlabel="Similaridade (0 a 1)",
    xlim_max=1.0,
    value_fmt="{:.3f}",
)


### Termos mais compartilhados

Os termos com maior frequência combinada nos dois documentos — o núcleo de
vocabulário que os dois textos têm efetivamente em comum.

In [ ]:
MIN_COMBINED_COUNT = 4

shared_terms = [
    (t, brics_counts[t] + ocde_counts[t])
    for t in vocab_intersection
    if brics_counts[t] + ocde_counts[t] >= MIN_COMBINED_COUNT
]
shared_terms.sort(key=lambda x: x[1], reverse=True)

plot_ranked_bar(
    shared_terms[:15],
    "Termos mais compartilhados entre os dois documentos (nº total de ocorrências)",
)


## 2. Divergência e convergência por termo

### Gráfico tornado — quem prioriza cada termo

Para os termos com pelo menos 4 ocorrências combinadas, comparamos a
frequência relativa (por 1.000 tokens) em cada documento. O gráfico mostra,
de um lado, os termos mais característicos do BRICS e, do outro, os mais
característicos da OCDE — ordenados pela diferença entre os dois.

In [ ]:
CATEGORICAL_BLUE = "#2a78d6"   # BRICS
CATEGORICAL_AQUA = "#1baf7a"   # OECD
TOP_N_EACH_SIDE = 12

rows = []
for t in vocab_union:
    b = relative_freq(brics_counts, len(brics_tokens), t)
    o = relative_freq(ocde_counts, len(ocde_tokens), t)
    combined_count = brics_counts.get(t, 0) + ocde_counts.get(t, 0)
    if combined_count >= MIN_COMBINED_COUNT:
        rows.append((t, b, o, b - o))

rows.sort(key=lambda x: x[3])
oecd_leaning = rows[:TOP_N_EACH_SIDE]                 # diferença mais negativa -> OCDE prioriza mais
brics_leaning = rows[-TOP_N_EACH_SIDE:]               # diferença mais positiva -> BRICS prioriza mais
tornado_rows = oecd_leaning + brics_leaning
tornado_rows.sort(key=lambda x: x[3])                 # OCDE no topo, BRICS na base

labels = [r[0] for r in tornado_rows]
brics_vals = [r[1] for r in tornado_rows]
ocde_vals = [r[2] for r in tornado_rows]

fig, ax = plt.subplots(figsize=(9, 0.4 * len(tornado_rows) + 1), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = range(len(tornado_rows))
ax.barh(y_pos, [-v for v in brics_vals], color=CATEGORICAL_BLUE, height=0.6, zorder=3, label=BRICS_LABEL)
ax.barh(y_pos, ocde_vals, color=CATEGORICAL_AQUA, height=0.6, zorder=3, label=OECD_LABEL)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9)
ax.axvline(0, color=GRIDLINE, linewidth=1.0, zorder=2)

ax.set_xlabel("Frequência relativa (ocorrências por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title(
    "Termos que cada documento prioriza — BRICS (esquerda) × OCDE (direita)",
    color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14,
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)

max_abs = max(max(brics_vals), max(ocde_vals))
ax.set_xlim(-max_abs * 1.15, max_abs * 1.15)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.0f}" for x in xticks], color=INK_MUTED, fontsize=9)

ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### Dispersão BRICS × OCDE — convergência e divergência

Cada ponto é um termo (com pelo menos 4 ocorrências combinadas), posicionado
pela frequência relativa no BRICS (eixo X) e na OCDE (eixo Y). A linha
diagonal tracejada marca "mesma prioridade nos dois documentos": pontos
próximos a ela **convergem**; pontos afastados da diagonal — grudados em um
dos eixos — **divergem**, revelando o vocabulário específico de cada
documento.

In [ ]:
CATEGORICAL_YELLOW = "#eda100"  # termos convergentes (equilíbrio entre os dois documentos)
BALANCE_RATIO_THRESHOLD = 0.6    # min(b, o) / max(b, o) >= 0.6 => convergente

scatter_rows = [r for r in rows]  # já filtrado por MIN_COMBINED_COUNT acima

def classify(b, o):
    ratio = min(b, o) / max(b, o) if max(b, o) > 0 else 1.0
    if ratio >= BALANCE_RATIO_THRESHOLD:
        return "convergente"
    return "BRICS" if b > o else "OCDE"

groups = {"convergente": [], "BRICS": [], "OCDE": []}
for t, b, o, _ in scatter_rows:
    groups[classify(b, o)].append((t, b, o))

fig, ax = plt.subplots(figsize=(8, 8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

max_val = max(max(b, o) for _, b, o, _ in scatter_rows) * 1.1
ax.plot([0, max_val], [0, max_val], linestyle="--", linewidth=1.2, color=INK_MUTED, zorder=2)

color_map = {"BRICS": CATEGORICAL_BLUE, "OCDE": CATEGORICAL_AQUA, "convergente": CATEGORICAL_YELLOW}
label_map = {"BRICS": BRICS_LABEL, "OCDE": OECD_LABEL, "convergente": "Convergente (equilíbrio)"}

for group_name in ["convergente", "BRICS", "OCDE"]:
    pts = groups[group_name]
    if not pts:
        continue
    xs = [p[1] for p in pts]
    ys = [p[2] for p in pts]
    ax.scatter(xs, ys, s=46, color=color_map[group_name], alpha=0.85, zorder=3,
               edgecolors=SURFACE, linewidths=0.6, label=label_map[group_name])

# Rotula os termos mais extremos em cada grupo para dar contexto ao gráfico
def top_extreme(pts, key, n=4):
    return sorted(pts, key=key, reverse=True)[:n]

annotate_targets = (
    top_extreme(groups["BRICS"], key=lambda p: p[1] - p[2])
    + top_extreme(groups["OCDE"], key=lambda p: p[2] - p[1])
    + top_extreme(groups["convergente"], key=lambda p: p[1] + p[2])
)
for t, x, y in annotate_targets:
    ax.annotate(t, (x, y), textcoords="offset points", xytext=(6, 4),
                fontsize=8, color=INK_PRIMARY)

ax.set_xlabel(f"Frequência relativa — {BRICS_LABEL} (por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_ylabel(f"Frequência relativa — {OECD_LABEL} (por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title("Convergência e divergência de termos — BRICS × OCDE", color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

ax.set_xlim(0, max_val)
ax.set_ylim(0, max_val)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color(GRIDLINE)
ax.spines["bottom"].set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)

ax.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

ax.text(max_val * 0.02, max_val * 0.97, "↑ mais enfatizado pela OCDE", color=INK_MUTED, fontsize=9, va="top")
ax.text(max_val * 0.97, max_val * 0.03, "→ mais enfatizado pelo BRICS", color=INK_MUTED, fontsize=9, ha="right")

ax.legend(loc="upper left", frameon=False, labelcolor=INK_PRIMARY, fontsize=9, bbox_to_anchor=(0.02, 0.9))

fig.tight_layout()
plt.show()


## Síntese

- A similaridade de vocabulário bruta (Jaccard) é moderada-baixa e a
  similaridade de cosseno (que pondera pela frequência) é moderada — os dois
  documentos tratam do mesmo tema (governança de IA), mas com vocabulário e
  ênfases distintos.
- O **BRICS** prioriza termos ligados a desenvolvimento, países em
  desenvolvimento, economia digital e cooperação internacional
  (*countries*, *digital*, *development*, *developing*, *economy*,
  *technologies*, *innovation*, *international*, *governance*).
- A **OCDE** prioriza termos ligados à arquitetura de governança do próprio
  sistema de IA e aos atores responsáveis por ele (*system*, *lifecycle*,
  *actors*, *stakeholders*, *governments*, *trustworthy*, *policy*,
  *recommendation*, *principles*).
- Termos como *development*, *systems*, *data*, *digital*, *rights* e
  *governance* aparecem com peso relevante nos dois textos e formam o núcleo
  comum de vocabulário entre as duas declarações.